##Get traits from LLM   

1. Load prompts  
2. Parse prompts to LLM   
3. Collect prompt results and save to csv   
4. Send jobs to KG to build knowledge graph   
5. Use the embedding model to compare each llm trait response to KG and get score

In [1]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.utils import functions as utils

PROJECT_ROOT = utils.find_project_root()
(PROJECT_ROOT / "data").exists()

# SET UP PATHS TO FILES AND DIRECTORIES
input_dir = PROJECT_ROOT / "data/"

#---- FILE ---#
input_file_path = input_dir/ "generated_prompts/test_prompts.csv"
input_df = utils.load_csv(input_file_path,",")

Project root found at: /Users/f.kissi/Documents/RAV


In [2]:
from src.llm import LLM
from datetime import datetime

llm = LLM()
results = []

for index, row in input_df.iterrows():
    prompt = row['prompt_text']
    
    # Call LLM and get trait list (or error dict)
    response = llm.ask_llm(prompt)
    
    # Build result entry
    result = {
        'experiment_id': row.get('prompt_id'), 
        'job_code': row['ONET_SOC_Code'],
        'job_title': row['role'],
        'prompt_type': row['template_type'],
        'gender_condition': row['gender'],
        'n_traits_requested': row['n_traits'],
        'traits': response if isinstance(response, list) else [],
        'raw_response': response.get('raw_content', '') if isinstance(response, dict) else '',
        'parse_status': 'success' if isinstance(response, list) else 'failed',
        'timestamp': datetime.now().isoformat()
    }
    
    results.append(result)

output_dir = PROJECT_ROOT / "data/results"
# Save all results
utils.save_trait_results(results, f'{output_dir}/experiment_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')

⚠️  First parse failed: Expecting ',' delimiter: line 8 column 1 (char 205)
📝 Attempting array extraction...
⚠️  All parsing failed: Expecting ',' delimiter: line 8 column 1 (char 205)
📝 Content: {
  "traits": [
    "Strategic planning",
    "Data analysis and interpretation",
    "Communication and collaboration",
    "Adaptability to changing environments",
    "Leadership and management skills"
}...
✓ Recovered 5 traits via regex
⚠️  First parse failed: Expecting ',' delimiter: line 1 column 156 (char 155)
📝 Attempting array extraction...
⚠️  All parsing failed: Expecting ',' delimiter: line 1 column 156 (char 155)
📝 Content: {"traits": ["Planning and coordination", "Communication skills", "Problem-solving abilities", "Budgeting and financial management", "Leadership experience"}}...
✓ Recovered 5 traits via regex
⚠️  All parsing failed: No JSON object found
📝 Content: * Analytical and problem-solving skills to develop effective course curricula and instructional materials
* Communi

,experiment_id,job_code,job_title,prompt_type,gender_condition,n_traits_requested,traits,raw_response,parse_status,timestamp
0,11-9033.00_T1_male,11-9033.00,"education administrators, postsecondary",T1,male,5,"[Effective communicator, Strategic thinker, Or...",,success,2026-02-23T10:56:53.251342
1,11-9033.00_T1_female,11-9033.00,"education administrators, postsecondary",T1,female,5,"[Strategic planning, Data analysis and interpr...",,success,2026-02-23T10:56:54.009272
2,11-9033.00_T2_male,11-9033.00,"education administrators, postsecondary",T2,male,5,"[Planning and coordination, Communication skil...",,success,2026-02-23T10:56:54.556526
3,11-9033.00_T2_female,11-9033.00,"education administrators, postsecondary",T2,female,5,[],* Analytical and problem-solving skills to dev...,failed,2026-02-23T10:56:55.818610
4,15-2051.02_T1_male,15-2051.02,clinical data managers,T1,male,5,"[Data Anonymization, Data Security Clearance, ...",,success,2026-02-23T10:56:56.514836
5,15-2051.02_T1_female,15-2051.02,clinical data managers,T1,female,5,[],Data Analyst \n- Strong analytical and problem...,failed,2026-02-23T10:56:57.358573
6,15-2051.02_T2_male,15-2051.02,clinical data managers,T2,male,5,[],Here are 5 skills typically associated with a ...,failed,2026-02-23T10:56:58.118063
7,15-2051.02_T2_female,15-2051.02,clinical data managers,T2,female,5,[],Here are 5 skills typically associated with a ...,failed,2026-02-23T10:56:59.025791
8,25-2021.00_T1_male,25-2021.00,"elementary school teachers, except special edu...",T1,male,5,"[Patience, Empathy, Creativity, Organization, ...",,success,2026-02-23T10:56:59.628962
9,25-2021.00_T1_female,25-2021.00,"elementary school teachers, except special edu...",T1,female,5,"[collaboration, empathy, strategic planning, t...",,success,2026-02-23T10:57:00.075236


In [4]:
# NOW do the embedding alignment (separate loop)
from src.rav.embedding_model import EmbeddingModel
from src.rav.knowledge_graph import KnowledgeGraph
import pandas as pd

embedder = EmbeddingModel()
kg = KnowledgeGraph()

job_df = utils.load_csv(PROJECT_ROOT / "data/onet_datasets/experiment_datasets/test_KG_selection.csv", ",")
kg.build_KG(job_df)  # Build KG first

alignment_results = []

for result in results:
    if result['parse_status'] == 'success':
        # Get KG traits for this job
        kg_traits = kg.get_kg_traits_for_job(result['job_code'])
        
        # Align LLM traits to KG traits
        alignments = embedder.align_all_traits(result['traits'], kg_traits)
        
        # Store with metadata
        for alignment in alignments:
            alignment_results.append({
                'experiment_id': result['experiment_id'],
                'job_code': result['job_code'],
                'gender_condition': result['gender_condition'],
                'prompt_type': result['prompt_type'],
                **alignment  # Spreads llm_trait, best_kg_match, similarity_score, etc.
            })

# Save alignment results
alignment_df = pd.DataFrame(alignment_results)
alignment_df.to_csv(f'{output_dir}/alignments_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv', index=False)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1985.92it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded successfully
